# Huấn luyện và đánh giá M2/M3

M1 đã hoàn tất. Hai nhánh khởi đầu từ cùng checkpoint-3683: M2 học chosen, M3 học chosen/rejected.

Nghiên cứu chính dùng 1.000 cặp, cùng ngân sách ba epoch cho M2/M3. Các mục đầu giữ lượt 186 cặp để tra cứu lịch sử; không chạy lại. Kết quả và việc đang làm ghi trong TIEN_DO.md.

Không dùng Chạy tất cả. Sau khi mount Drive, chỉ chạy setup và ô thực thi của đúng giai đoạn cần tiếp tục.

[M1: code và output đã chạy](https://colab.research.google.com/drive/13Abjl7kJyDKzbxyYsCetJ540OzdK2P_E). Bản local: notebooks/archive/03_m1_completed_with_outputs_20261013.ipynb.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
ROOT = Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')
if not ROOT.is_dir():
    raise FileNotFoundError('Project folder is missing from this Drive')
LAUNCH_PATH = ROOT / 'source/preference_training_launch_v1.json'
if not LAUNCH_PATH.is_file():
    raise FileNotFoundError(f'Launch record is missing: {LAUNCH_PATH}')
print('Project storage ready:', ROOT)


In [ ]:
launch = json.loads(LAUNCH_PATH.read_text(encoding='utf-8'))
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
bootstrap_sha = '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != bootstrap_sha:
    raise ValueError('Bootstrap source changed')
release = launch['source_release']
if not release.startswith('source_') or any(char in release for char in '/\\:'):
    raise ValueError('Invalid source release name')
command = [sys.executable, str(bootstrap), '--archive',
           str(ROOT / 'source/releases' / f'{release}.zip'),
           '--sha256', launch['source_sha256'], '--work-root', '/content/vqa_sources']
REPO = Path(subprocess.check_output(command, text=True).strip())
sys.path.insert(0, str(REPO / 'src'))
from vqa_explanation.utils.process import run_command  # noqa: E402

ENV_DIR = Path('/content/vqa_envs') / release
PYTHON = ENV_DIR / 'bin/python'
print('Source:', release)
run_command([sys.executable, str(REPO / 'scripts/setup_preference_environment.py'),
             '--environment-dir', str(ENV_DIR)], cwd=REPO)
print('Training interpreter:', PYTHON)


In [ ]:
from vqa_explanation.training.preference_launch import launch_command, stage_output
from vqa_explanation.utils.environment import load_environment

MODE = 'inspect'
RUN_FOLDER = ROOT / launch['run_group']
RESUME = {'m2': None, 'm3': None}
ATTEMPT = {'m2': 'initial', 'm3': 'initial'}

if launch.get('data_locked') is True:
    environment = load_environment(REPO / launch['environment'])
    if not environment['image_root'].exists():
        bundle = ROOT / launch['dataset_bundle']
        digest = hashlib.sha256((bundle / 'bundle.json').read_bytes()).hexdigest()
        if digest != launch['dataset_bundle_sha256']:
            raise ValueError('Dataset bundle changed')
        run_command([str(PYTHON), str(REPO / 'scripts/verify_coco_bundle.py'),
                        str(bundle), '--extract-to', str(environment['image_root'])],
                    cwd=REPO)

for arm in ('m2', 'm3'):
    command = launch_command(REPO, LAUNCH_PATH, ROOT, arm, 'inspect', RUN_FOLDER,
                             python_executable=PYTHON)
    run_command(command, cwd=REPO)


In [ ]:
command = launch_command(REPO, LAUNCH_PATH, ROOT, 'm2', MODE, RUN_FOLDER,
                         RESUME['m2'], ATTEMPT['m2'], python_executable=PYTHON)
run_command(command, cwd=REPO)


In [ ]:
command = launch_command(REPO, LAUNCH_PATH, ROOT, 'm3', MODE, RUN_FOLDER,
                         RESUME['m3'], ATTEMPT['m3'], python_executable=PYTHON)
run_command(command, cwd=REPO)


In [ ]:
for arm in ('m2', 'm3'):
    result = stage_output(RUN_FOLDER, arm, MODE, ATTEMPT[arm]) / 'training_result.json'
    if result.is_file():
        report = json.loads(result.read_text(encoding='utf-8'))
        fields = ('status', 'global_step', 'epoch', 'training_loss', 'peak_vram_gib',
                  'policy_updated', 'reference_unchanged', 'saved_policy_verified')
        print(arm, json.dumps({key: report.get(key) for key in fields}, indent=2))
    else:
        print(arm, 'No training result:', result)


## Đánh giá validation

Chạy sau khi M2/M3 hoàn tất. Xuất đủ 1.407 câu cho bốn nhánh trong cùng môi trường; không train lại. Kết quả cũ giữ nguyên. CLI kiểm checkpoint và lưu từng câu để có thể nối tiếp.


In [ ]:
EVAL_LAUNCH = json.loads((ROOT / 'source/validation_launch_v1.json').read_text('utf-8'))
eval_release = EVAL_LAUNCH['source_release']
if not eval_release.startswith('source_') or any(char in eval_release for char in '/\\:'):
    raise ValueError('Invalid evaluation source name')
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != bootstrap_sha:
    raise ValueError('Bootstrap source changed')
EVAL_REPO = Path(subprocess.check_output([
    sys.executable, str(bootstrap), '--archive',
    str(ROOT / 'source/releases' / f'{eval_release}.zip'),
    '--sha256', EVAL_LAUNCH['source_sha256'], '--work-root', '/content/vqa_sources',
], text=True).strip())
run_command([sys.executable, '-m', 'pip', '--python', str(PYTHON),
             'install', '--no-deps', '-e', str(EVAL_REPO)], cwd=EVAL_REPO)
run_command([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'check'], cwd=EVAL_REPO)
VALIDATION_COMMAND = [
    str(PYTHON), '-u', str(EVAL_REPO / 'scripts/run_study_validation.py'),
    '--environment', str(EVAL_REPO / launch['environment']),
    '--run-group', str(ROOT / launch['run_group']),
    '--m1-adapter', str(ROOT / launch['m1_adapter']),
    '--output-dir', str(ROOT / EVAL_LAUNCH['output_dir']),
]
if EVAL_LAUNCH.get('relocate_from'):
    VALIDATION_COMMAND += ['--relocate-from', str(ROOT / EVAL_LAUNCH['relocate_from'])]


In [ ]:
run_command(VALIDATION_COMMAND + ['--inspect'], cwd=EVAL_REPO)


In [ ]:
run_command(VALIDATION_COMMAND, cwd=EVAL_REPO)


## Mở rộng bộ cặp lên 1.000

Kiểm lại 186 cặp cũ, dùng ứng viên M1 đã lưu rồi bổ sung ảnh train mới. Chạy setup → kiểm đầu vào → xử lý cặp. Giữ nguyên output và checkpoint của lượt trước; phần này không train M2/M3.


In [ ]:
assert ROOT.is_dir(), 'Project storage is not mounted'
print('Project storage:', ROOT.resolve())
print('Training interpreter:', PYTHON, PYTHON.is_file())
storage_keys = ('preferences', 'm1_adapter', 'm1_selection', 'environment', 'dataset_bundle')
print(json.dumps({key: launch[key] for key in storage_keys}, indent=2))


In [ ]:
import yaml

EXPANSION_LAUNCH = json.loads((ROOT / 'source/preference_expansion_v2.json').read_text('utf-8'))
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != bootstrap_sha:
    raise ValueError('Bootstrap source changed')
EXPANSION_REPO = Path(subprocess.check_output([
    sys.executable, str(bootstrap), '--archive',
    str(ROOT / 'source/releases' / f"{EXPANSION_LAUNCH['source_release']}.zip"),
    '--sha256', EXPANSION_LAUNCH['source_sha256'], '--work-root', '/content/vqa_sources',
], text=True).strip())
EXPANSION_SCRIPT = EXPANSION_REPO / 'scripts/expand_preferences.py'
EXPANSION_INPUTS = Path(subprocess.check_output([
    str(PYTHON), str(EXPANSION_SCRIPT), 'restore', '--archive',
    str(ROOT / EXPANSION_LAUNCH['metadata_bundle']), '--sha256',
    EXPANSION_LAUNCH['metadata_sha256'], '--work-root', '/content/vqa_metadata',
], text=True).strip())
expansion_environment = yaml.safe_load((REPO / launch['environment']).read_text('utf-8'))
WORK_IMAGES = Path(expansion_environment['image_root'])
if not WORK_IMAGES.exists():
    dataset_bundle = ROOT / launch['dataset_bundle']
    bundle_sha = hashlib.sha256((dataset_bundle / 'bundle.json').read_bytes()).hexdigest()
    if bundle_sha != launch['dataset_bundle_sha256']:
        raise ValueError('Dataset bundle changed')
    run_command([str(PYTHON), str(REPO / 'scripts/verify_coco_bundle.py'),
                 str(dataset_bundle), '--extract-to', str(WORK_IMAGES)], cwd=REPO)
EXPANSION_COMMAND = [str(PYTHON), '-u', str(EXPANSION_SCRIPT), 'run',
    '--annotations', str(WORK_IMAGES / 'metadata/all.jsonl'),
    '--image-root', str(WORK_IMAGES),
    '--old-pairs', str(EXPANSION_INPUTS / 'old_pairs/accepted.jsonl'),
    '--candidates', str(EXPANSION_INPUTS / 'candidates.jsonl'),
    '--selection', str(EXPANSION_INPUTS / 'pool'),
    '--m1-adapter', str(ROOT / launch['m1_adapter']),
    '--m1-selection', str(ROOT / launch['m1_selection']),
    '--output-dir', str(ROOT / EXPANSION_LAUNCH['output_dir']), '--target', '1000']
if EXPANSION_LAUNCH.get('teacher'):
    EXPANSION_COMMAND += ['--teacher-id', EXPANSION_LAUNCH['teacher']['id'],
                          '--teacher-revision', EXPANSION_LAUNCH['teacher']['revision']]
print('Expansion source:', EXPANSION_REPO)
print('Metadata:', EXPANSION_INPUTS)


In [ ]:
run_command(EXPANSION_COMMAND + ['--inspect'], cwd=EXPANSION_REPO)


In [ ]:
ITEM_LIMIT = 20  # Set to zero after checking the first batch.
run_command(EXPANSION_COMMAND + ['--max-items', str(ITEM_LIMIT)], cwd=EXPANSION_REPO)


## M2/M3 — 1,000 reviewed pairs
Use the reviewed dataset locked on 15 October. Run setup, preflight and each arm's GPU gate before training. Each arm starts from M1-3683 for three epochs, seed 42. Keep the earlier outputs unchanged.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import yaml

PROJECT_ROOTS = (
    Path('/content/drive/.shortcut-targets-by-id/17bcTXATT45J5LEmtruMRs1ddwRY5kBwc/Vietnamese_VQA_NLE'),
    Path('/content/drive/MyDrive/Vietnamese_VQA_NLE'),
)
ROOT = next((path for path in PROJECT_ROOTS
    if (path / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Mount the project folder containing the frozen M1 selection')
LAUNCH_PATH_V2 = ROOT / 'source/preference_training_launch_v2_20261015.json'
if not LAUNCH_PATH_V2.is_file():
    raise FileNotFoundError(LAUNCH_PATH_V2)
launch_v2 = json.loads(LAUNCH_PATH_V2.read_text(encoding='utf-8'))
assert launch_v2.get('data_locked') is True and launch_v2.get('expected_pairs') == 1000
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
bootstrap_sha256 = '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != bootstrap_sha256:
    raise ValueError('Bootstrap checksum mismatch')
source_archive = ROOT / 'source/releases' / f"{launch_v2['source_release']}.zip"
REPO_V2 = Path(subprocess.check_output([sys.executable, str(bootstrap), '--archive',
    str(source_archive), '--sha256', launch_v2['source_sha256'],
    '--work-root', '/content/vqa_sources'], text=True).strip())
sys.path.insert(0, str(REPO_V2 / 'src'))
from vqa_explanation.training.preference_launch import launch_command, stage_output  # noqa: E402
from vqa_explanation.utils.environment import load_environment  # noqa: E402
from vqa_explanation.utils.process import run_command  # noqa: E402

study_v2 = yaml.safe_load((REPO_V2 / launch_v2['study_config']).read_text('utf-8'))
if study_v2['preferences'].get('data_locked') is not True:
    raise ValueError('Dataset quality review is incomplete; training remains paused')
ENV_DIR_V2 = Path('/content/vqa_envs') / launch_v2['source_release']
run_command([sys.executable, str(REPO_V2 / 'scripts/setup_preference_environment.py'),
    '--environment-dir', str(ENV_DIR_V2)], cwd=REPO_V2)
PYTHON_V2 = ENV_DIR_V2 / 'bin/python'
run_command([str(PYTHON_V2), str(REPO_V2 / 'scripts/restore_preference_bundle.py'),
    '--archive', str(ROOT / launch_v2['preference_archive']),
    '--sha256', launch_v2['preference_archive_sha256'],
    '--destination', str(ROOT / launch_v2['preference_root'])], cwd=REPO_V2)
RUN_FOLDER_V2 = ROOT / launch_v2['run_group']
print('Source:', launch_v2['source_release'])
print('Run folder:', RUN_FOLDER_V2)


In [ ]:
environment_v2 = load_environment(REPO_V2 / launch_v2['environment'])
if not environment_v2['image_root'].is_dir():
    bundle = ROOT / launch_v2['dataset_bundle']
    digest = hashlib.sha256((bundle / 'bundle.json').read_bytes()).hexdigest()
    if digest != launch_v2['dataset_bundle_sha256']:
        raise ValueError('Dataset bundle checksum mismatch')
    run_command([str(PYTHON_V2), str(REPO_V2 / 'scripts/verify_coco_bundle.py'),
        str(bundle), '--extract-to', str(environment_v2['image_root'])], cwd=REPO_V2)
for arm in ('m2', 'm3'):
    command = launch_command(REPO_V2, LAUNCH_PATH_V2, ROOT, arm, 'inspect',
        RUN_FOLDER_V2, python_executable=PYTHON_V2)
    run_command(command, cwd=REPO_V2)


In [ ]:
command = launch_command(REPO_V2, LAUNCH_PATH_V2, ROOT, 'm2', 'gate',
    RUN_FOLDER_V2, python_executable=PYTHON_V2)
run_command(command, cwd=REPO_V2)


In [ ]:
command = launch_command(REPO_V2, LAUNCH_PATH_V2, ROOT, 'm2', 'train',
    RUN_FOLDER_V2, python_executable=PYTHON_V2)
run_command(command, cwd=REPO_V2)


In [ ]:
command = launch_command(REPO_V2, LAUNCH_PATH_V2, ROOT, 'm3', 'gate',
    RUN_FOLDER_V2, python_executable=PYTHON_V2)
run_command(command, cwd=REPO_V2)


In [ ]:
command = launch_command(REPO_V2, LAUNCH_PATH_V2, ROOT, 'm3', 'train',
    RUN_FOLDER_V2, python_executable=PYTHON_V2)
run_command(command, cwd=REPO_V2)


In [ ]:
for arm in ('m2', 'm3'):
    result_path = stage_output(RUN_FOLDER_V2, arm, 'train') / 'training_result.json'
    if result_path.is_file():
        result = json.loads(result_path.read_text(encoding='utf-8'))
        print(arm.upper(), result['status'], 'steps=', result['global_step'],
              'epochs=', result['epoch'], 'hours=', round(result['estimated_full_train_hours'], 2))
    else:
        print(arm.upper(), 'not run yet')


## Validation — 1,000-pair study

After both arms finish: mount Drive → setup below → inspect → export. After a disconnect, repeat only these steps; completed predictions are verified and skipped. Use all 1,407 validation questions for M0–M3; keep test closed. Earlier validation outputs belong to the 186-pair run.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

PROJECT_ROOTS = (
    Path('/content/drive/.shortcut-targets-by-id/17bcTXATT45J5LEmtruMRs1ddwRY5kBwc/Vietnamese_VQA_NLE'),
    Path('/content/drive/MyDrive/Vietnamese_VQA_NLE'),
)
ROOT = next((path for path in PROJECT_ROOTS
    if (path / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Mount the project Drive with the frozen M1 selection first')
LAUNCH_PATH_V2 = ROOT / 'source/preference_training_launch_v2_20261015.json'
validation_record_path = ROOT / 'source/preference_validation_v2_resume01.json'
validation_record = json.loads(validation_record_path.read_text('utf-8'))
if validation_record.get('schema') != 'preference_validation_launch_v1':
    raise ValueError('Unexpected validation launch')
launch_digest = hashlib.sha256(LAUNCH_PATH_V2.read_bytes()).hexdigest()
if launch_digest != validation_record['training_launch_sha256']:
    raise ValueError('Training launch changed')
launch_v2 = json.loads(LAUNCH_PATH_V2.read_text('utf-8'))
if launch_v2.get('data_locked') is not True or launch_v2.get('expected_pairs') != 1000:
    raise ValueError('Require the locked 1,000-pair training launch')
if validation_record['run_group'] != launch_v2['run_group']:
    raise ValueError('Validation points to a different training run')
RUN_FOLDER_V2 = ROOT / launch_v2['run_group']
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
bootstrap_sha = '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != bootstrap_sha:
    raise ValueError('Bootstrap source changed')
EVAL_REPO_V2 = Path(subprocess.check_output([sys.executable, str(bootstrap),
    '--archive', str(ROOT / 'source/releases' / f"{validation_record['source_release']}.zip"),
    '--sha256', validation_record['source_sha256'], '--work-root', '/content/vqa_sources'],
    text=True).strip())
sys.path.insert(0, str(EVAL_REPO_V2 / 'src'))
from vqa_explanation.utils.environment import load_environment  # noqa: E402
from vqa_explanation.utils.process import run_command  # noqa: E402

EVAL_ENV_V2 = Path('/content/vqa_envs') / validation_record['source_release']
run_command([sys.executable, str(EVAL_REPO_V2 / 'scripts/setup_preference_environment.py'),
    '--environment-dir', str(EVAL_ENV_V2)], cwd=EVAL_REPO_V2)
eval_environment = load_environment(EVAL_REPO_V2 / launch_v2['environment'])
if not eval_environment['annotations'].is_file():
    bundle = ROOT / launch_v2['dataset_bundle']
    bundle_hash = hashlib.sha256((bundle / 'bundle.json').read_bytes()).hexdigest()
    if bundle_hash != launch_v2['dataset_bundle_sha256']:
        raise ValueError('Dataset bundle changed')
    run_command([str(EVAL_ENV_V2 / 'bin/python'),
        str(EVAL_REPO_V2 / 'scripts/verify_coco_bundle.py'),
        str(bundle), '--extract-to', str(eval_environment['image_root'])], cwd=EVAL_REPO_V2)
EVAL_COMMAND_V2 = [str(EVAL_ENV_V2 / 'bin/python'),
    str(EVAL_REPO_V2 / 'scripts/run_study_validation.py'),
    '--environment', str(EVAL_REPO_V2 / launch_v2['environment']),
    '--study-config', str(EVAL_REPO_V2 / launch_v2['study_config']),
    '--run-group', str(RUN_FOLDER_V2), '--m1-adapter', str(ROOT / launch_v2['m1_adapter']),
    '--output-dir', str(ROOT / validation_record['output_dir'])]


In [ ]:
run_command(EVAL_COMMAND_V2 + ['--inspect'], cwd=EVAL_REPO_V2)


In [ ]:
run_command(EVAL_COMMAND_V2, cwd=EVAL_REPO_V2)


## Image permutation — validation

Use the fixed 200-image bundle for M1/M2/M3, seed 42. Keep questions and generation settings unchanged; only swap images. This measures image sensitivity, not explanation correctness. Mount Drive first, then run setup and export below. Do not rerun training or use test.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

IMAGE_RELEASE = 'source_20261015_184626_136221'
IMAGE_SOURCE_SHA = '9b9540242072cb32b97229610a0ce4c141889027ad881b3675b1b0ff08178d02'
IMAGE_BUNDLE_SHA = '729a89c1d3f61dc8df67376c40f5fda0717b7a96b7bcd26e806a5066526c8f63'
image_launch_path = ROOT / 'source/preference_training_launch_v2_20261015.json'
assert hashlib.sha256(image_launch_path.read_bytes()).hexdigest() == (
    'b3eb6e6c8fe4615eb50253d76180a9ecdca8ebf413cd2f059dabb25c594f9938')
image_launch = json.loads(image_launch_path.read_text('utf-8'))
image_bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
assert hashlib.sha256(image_bootstrap.read_bytes()).hexdigest() == (
    '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40')
IMAGE_REPO = Path(subprocess.check_output([sys.executable, str(image_bootstrap),
    '--archive', str(ROOT / 'source/releases' / f'{IMAGE_RELEASE}.zip'),
    '--sha256', IMAGE_SOURCE_SHA, '--work-root', '/content/vqa_sources'], text=True).strip())
sys.path.insert(0, str(IMAGE_REPO / 'src'))
from vqa_explanation.utils.process import run_command  # noqa: E402

IMAGE_ENV = Path('/content/vqa_envs') / IMAGE_RELEASE
IMAGE_PYTHON = IMAGE_ENV / 'bin/python'
run_command([sys.executable, str(IMAGE_REPO / 'scripts/setup_preference_environment.py'),
    '--environment-dir', str(IMAGE_ENV)], cwd=IMAGE_REPO)
image_environment = IMAGE_REPO / image_launch['environment']
image_paths = json.loads(subprocess.check_output([str(IMAGE_PYTHON), '-c',
    'import json,sys; from pathlib import Path; '
    'from vqa_explanation.utils.environment import load_environment; '
    'print(json.dumps({k:str(v) for k,v in load_environment(Path(sys.argv[1])).items()}))',
    str(image_environment)], cwd=IMAGE_REPO, text=True))
if not Path(image_paths['annotations']).is_file():
    image_dataset = ROOT / image_launch['dataset_bundle']
    dataset_digest = hashlib.sha256((image_dataset / 'bundle.json').read_bytes()).hexdigest()
    assert dataset_digest == image_launch['dataset_bundle_sha256']
    run_command([str(IMAGE_PYTHON), str(IMAGE_REPO / 'scripts/verify_coco_bundle.py'),
        str(image_dataset), '--extract-to', image_paths['image_root']], cwd=IMAGE_REPO)
IMAGE_INPUT = ROOT / 'evaluation_runs/image_permutation/validation_200_seed42/input'
IMAGE_OUTPUT = ROOT / 'evaluation_runs/image_permutation/preferences_v2_seed42'
IMAGE_COMMAND = [str(IMAGE_PYTHON), '-u', str(IMAGE_REPO / 'scripts/check_image_dependence.py')]
IMAGE_OPTIONS = ['--environment', str(image_environment), '--bundle', str(IMAGE_INPUT)]
if not IMAGE_INPUT.exists():
    run_command(IMAGE_COMMAND + ['prepare'] + IMAGE_OPTIONS, cwd=IMAGE_REPO)
image_manifest_bytes = (IMAGE_INPUT / 'manifest.json').read_bytes().replace(b'\r\n', b'\n')
assert hashlib.sha256(image_manifest_bytes).hexdigest() == IMAGE_BUNDLE_SHA
image_evidence = json.loads(subprocess.check_output([str(IMAGE_PYTHON), '-c',
    'import json,sys; from vqa_explanation.inference.study_validation import prepare_validation; '
    'env,ann,adapters,evidence=prepare_validation(*sys.argv[1:5],study_config=sys.argv[5]); '
    'print(json.dumps(dict(adapters={k:str(v) for k,v in adapters.items() if v},'
    'evidence=evidence)))',
    str(image_environment), str(ROOT / image_launch['run_group']),
    str(ROOT / image_launch['m1_adapter']), str(IMAGE_OUTPUT),
    str(IMAGE_REPO / image_launch['study_config'])], cwd=IMAGE_REPO, text=True))
run_command([str(IMAGE_PYTHON), '-c',
    'import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'],
    cwd=IMAGE_REPO)
print('Ready: fixed 200 validation images; final M1/M2/M3 adapters verified.')


In [ ]:
for arm, experiment in (('m1', 'M1_SFT'), ('m2', 'M2_SFT_CHOSEN'), ('m3', 'M3_DPO')):
    print('Image diagnostic:', arm.upper(), flush=True)
    run_command(IMAGE_COMMAND + ['run'] + IMAGE_OPTIONS + [
        '--output-dir', str(IMAGE_OUTPUT / arm), '--experiment-id', experiment,
        '--adapter', image_evidence['adapters'][arm]], cwd=IMAGE_REPO)
print('Complete. Predictions and sensitivity scores:', IMAGE_OUTPUT)


## Lặp seed 43

Giữ bộ 1.000 cặp, M1-3683, LR 1e-5 và ba epoch. Chỉ đổi seed, lưu run riêng. Setup → M2 → M3; không chạy lại các mục lịch sử. Mỗi nhánh kiểm GPU năm update rồi train chính từ M1, không tiếp nối trọng số của lượt kiểm.


In [ ]:
REPLICATION_RECORD = ROOT / 'source/preference_training_seed43_20261016.json'
replication = json.loads(REPLICATION_RECORD.read_text('utf-8'))
assert replication['training_seed'] == 43
assert replication['study_config'] == 'configs/experiments/preference_study_seed43.yaml'
parent_record = ROOT / 'source/preference_training_launch_v2_20261015.json'
assert hashlib.sha256(parent_record.read_bytes()).hexdigest() == (
    'b3eb6e6c8fe4615eb50253d76180a9ecdca8ebf413cd2f059dabb25c594f9938')
parent_launch = json.loads(parent_record.read_text('utf-8'))
for key in ('preferences', 'preference_root', 'preference_archive',
            'preference_archive_sha256', 'm1_adapter', 'm1_selection',
            'environment', 'dataset_bundle', 'dataset_bundle_sha256',
            'preference_decision_record', 'preference_decision_record_sha256'):
    assert replication[key] == parent_launch[key], key
assert replication['run_group'] != parent_launch['run_group']
REPLICATION_RUN = ROOT / replication['run_group']
assert not REPLICATION_RUN.exists(), 'Run exists; inspect and resume it, do not start over'
replication_bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
assert hashlib.sha256(replication_bootstrap.read_bytes()).hexdigest() == (
    '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40')
REPLICATION_REPO = Path(subprocess.check_output([sys.executable,
    str(replication_bootstrap), '--archive',
    str(ROOT / 'source/releases' / f"{replication['source_release']}.zip"),
    '--sha256', replication['source_sha256'], '--work-root', '/content/vqa_sources'],
    text=True).strip())
REPLICATION_ENV = globals().get('IMAGE_ENV', Path('/content/vqa_envs/preference_seed43'))
run_command([sys.executable, str(REPLICATION_REPO / 'scripts/setup_preference_environment.py'),
    '--environment-dir', str(REPLICATION_ENV)], cwd=REPLICATION_REPO)
REPLICATION_PYTHON = REPLICATION_ENV / 'bin/python'
run_command([str(REPLICATION_PYTHON), '-c',
    'import sys,torch; from vqa_explanation.utils.environment import project_root; '
    'assert str(project_root()) == sys.argv[1]; assert torch.cuda.is_available(); '
    'print(torch.cuda.get_device_name(0))', str(REPLICATION_REPO)], cwd=REPLICATION_REPO)
run_command([str(REPLICATION_PYTHON),
    str(REPLICATION_REPO / 'scripts/restore_preference_bundle.py'),
    '--archive', str(ROOT / replication['preference_archive']),
    '--sha256', replication['preference_archive_sha256'],
    '--destination', str(ROOT / replication['preference_root'])], cwd=REPLICATION_REPO)
REPLICATION_COMMAND = [str(REPLICATION_PYTHON), '-u',
    str(REPLICATION_REPO / 'scripts/launch_preference_training.py'),
    '--launch-record', str(REPLICATION_RECORD), '--project-dir', str(ROOT)]
for arm in ('m2', 'm3'):
    run_command(REPLICATION_COMMAND + ['--arm', arm, '--mode', 'inspect'], cwd=REPLICATION_REPO)
print('Seed 43 ready:', REPLICATION_RUN)


In [ ]:
for mode in ('gate', 'train'):
    run_command(REPLICATION_COMMAND + ['--arm', 'm2', '--mode', mode,
        '--output-dir', str(REPLICATION_RUN / f'm2_{mode}')] +
        (['--gpu-gate', str(REPLICATION_RUN / 'm2_gate/training_result.json')]
         if mode == 'train' else []), cwd=REPLICATION_REPO)


In [ ]:
for mode in ('gate', 'train'):
    run_command(REPLICATION_COMMAND + ['--arm', 'm3', '--mode', mode,
        '--output-dir', str(REPLICATION_RUN / f'm3_{mode}')] +
        (['--gpu-gate', str(REPLICATION_RUN / 'm3_gate/training_result.json')]
         if mode == 'train' else []), cwd=REPLICATION_REPO)
